In [ ]:
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from core.genetic_algorithm import genetic_algorithm
from core.solver import solver

In [ ]:
instances = ["i04", "i06"]
all_results = {}
solver_results = {}
all_times = {}

In [ ]:
for instance in instances:
    instance_dir = Path("data") / instance
    results = []
    times = []

    solver_output_dir = instance_dir / "solutions" / "solver_solution.csv"
    ga_output_dir = instance_dir / "solutions" / "genetic_algorithm_solution.csv"

    start_solver = datetime.now(tz=UTC)
    solver_result = solver(instance_dir, solver_output_dir)
    end_solver = datetime.now(tz=UTC) - start_solver
    solver_seconds = end_solver.total_seconds()

    print(f"{instance} - Solver - Best Solution (fitness): {solver_result}")

    for _ in range(20):
        start_ga = datetime.now(tz=UTC)
        ga_result = genetic_algorithm(
            instance_dir,
            ga_output_dir,
            population_size=40,
            generations=1000,
            tournament_size=3,
            mutation_rate=0.2,
            elite_fraction=0.1,
            seed=datetime.now(UTC).timestamp(),
        )
        end_ga = datetime.now(tz=UTC) - start_ga
        ga_seconds = end_ga.total_seconds()

        print(f"{instance} - GA - Best Solution (fitness): {ga_result} | Time: {ga_seconds:.4f}s")
        results.append(ga_result)
        times.append(ga_seconds)

    all_results[instance] = results
    all_times[instance] = times

i04 - Solver - Best Solution (fitness): 32.0
i04 - GA - Best Solution (fitness): 84.0 | Time: 21.5396s
i04 - GA - Best Solution (fitness): 81.0 | Time: 21.6904s
i04 - GA - Best Solution (fitness): 79.0 | Time: 21.7838s
i04 - GA - Best Solution (fitness): 97.0 | Time: 21.6880s
i04 - GA - Best Solution (fitness): 90.0 | Time: 21.8609s
i04 - GA - Best Solution (fitness): 85.0 | Time: 22.0220s
i04 - GA - Best Solution (fitness): 91.0 | Time: 22.0487s
i04 - GA - Best Solution (fitness): 78.0 | Time: 22.0591s
i04 - GA - Best Solution (fitness): 79.0 | Time: 21.8174s
i04 - GA - Best Solution (fitness): 99.0 | Time: 21.8077s
i04 - GA - Best Solution (fitness): 82.0 | Time: 21.5371s
i04 - GA - Best Solution (fitness): 89.0 | Time: 21.8021s
i04 - GA - Best Solution (fitness): 86.0 | Time: 21.8444s
i04 - GA - Best Solution (fitness): 86.0 | Time: 21.6508s
i04 - GA - Best Solution (fitness): 90.0 | Time: 21.6071s
i04 - GA - Best Solution (fitness): 82.0 | Time: 21.6708s
i04 - GA - Best Solution (f

In [ ]:
images_dir = Path("images")
images_dir.mkdir(exist_ok=True)

In [ ]:
for instance, vals in all_results.items():
    y = np.array(vals, dtype=float)
    x = np.arange(1, len(y) + 1)

    mean = y.mean()
    std = y.std(ddof=1) if len(y) > 1 else 0.0

    plt.figure(figsize=(6, 4))

    plt.plot(x, y, marker="o", linewidth=1)
    plt.axhline(mean, linewidth=1)
    plt.fill_between([x.min(), x.max()], mean - std, mean + std, alpha=0.2)

    if instance in solver_results:
        plt.axhline(float(solver_results[instance]), linestyle="--", linewidth=1)

    plt.title(f"{instance} | média={mean:.4f} | std={std:.4f}")
    plt.xlabel("Execução")
    plt.ylabel("Best fitness (final)")
    plt.tight_layout()

    plt.savefig(images_dir / f"{instance}_execucoes_media_estabilidade.pdf")
    plt.close()

In [ ]:
data = [np.array(all_results[i], dtype=float) for i in instances]

plt.figure(figsize=(6, 4))
plt.boxplot(data, labels=instances, showmeans=True)
plt.ylabel("Best fitness (final)")
plt.title("Distribuição do GA por instância (estabilidade)")
plt.tight_layout()

plt.savefig(images_dir / "boxplot_estabilidade.pdf")
plt.close()

In [ ]:
for instance, times in all_times.items():
    y = np.array(times, dtype=float)
    x = np.arange(1, len(y) + 1)

    mean_t = y.mean()
    std_t = y.std(ddof=1) if len(y) > 1 else 0.0

    plt.figure(figsize=(6, 4))

    plt.plot(x, y, marker="o", color="orange", linewidth=1, label="Tempo por Execução")
    plt.axhline(mean_t, color="red", linewidth=1, label=f"Média: {mean_t:.4f}s")
    plt.fill_between([x.min(), x.max()], mean_t - std_t, mean_t + std_t, color="orange", alpha=0.2)

    plt.title(f"{instance} - Desempenho Temporal | média={mean_t:.4f}s")
    plt.xlabel("Execução")
    plt.ylabel("Tempo de execução (segundos)")
    plt.legend()
    plt.tight_layout()

    plt.savefig(images_dir / f"{instance}_tempo_estabilidade.pdf")
    plt.close()

In [ ]:
for instance in instances:
    instance_dir = Path("data") / instance

    solver_output_dir = instance_dir / "solutions" / "solver_solution.csv"
    ga_output_dir = instance_dir / "solutions" / "genetic_algorithm_solution.csv"

    solver_result = solver(instance_dir, solver_output_dir)
    print(f"{instance} - Solver - Best Solution (fitness): {solver_result}")

    results_a = []
    results_b = []

    for _ in range(5):
        result_a = genetic_algorithm(
            instance_dir,
            ga_output_dir,
            population_size=40,
            generations=1000,
            tournament_size=3,
            mutation_rate=0.2,
            elite_fraction=0.1,
            seed=datetime.now(tz=UTC).timestamp(),
        )

        result_b = genetic_algorithm(
            instance_dir,
            ga_output_dir,
            population_size=80,
            generations=1000,
            tournament_size=5,
            mutation_rate=0.4,
            elite_fraction=0.2,
            seed=datetime.now(tz=UTC).timestamp(),
        )

        print(f"{instance} - Config A: {result_a}")
        print(f"{instance} - Config B: {result_b}")

        results_a.append(result_a)
        results_b.append(result_b)

    A = np.array(results_a, dtype=float)
    B = np.array(results_b, dtype=float)

    plt.figure(figsize=(7,4))
    plt.boxplot([A, B], labels=["Config A", "Config B"], showmeans=True)

    plt.axhline(solver_result,linestyle="--",linewidth=1,label="Solver")
    plt.legend()

    plt.ylabel("Best fitness (final)")
    plt.title(f"{instance} | Comparação de Configurações do GA")
    plt.tight_layout()

    plt.savefig(images_dir / f"{instance}_comparacao_configuracoes.pdf")
    plt.close()